# 2.4 Edge Deployment Feasibility

Benchmark the **two-tier detection system** for edge deployment: measure inference latency, memory footprint, and throughput in Python (native LightGBM + ONNX Runtime), then export ONNX for Rust/WASM benchmarking.

**Models benchmarked:**
- **Tier 1 — Request-level** LGBM (36 features: 15 per-request + 21 IP-level session)
- **Tier 2 — Source-level** LGBM (13 features: IP-day behavioral profile)

**Target constraints:** <5ms inference, 50k req/s, WASM runtime, no GPU.

**Note:** TLS-granularity session features were removed due to label proxy leakage (see 2.3). The request-level model uses IP-level session features only, which capture genuine behavioral patterns. The source-level model aggregates per-request data into IP-day profiles for distributed attack detection (DDoS L7).

In [1]:
import os
os.chdir(os.path.join(os.path.dirname(os.path.abspath(".")), ""))
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import time
import json
import tracemalloc
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib
import onnxruntime as ort

from src.label_joining import join_labels
from src.features import compute_per_request_features, compute_session_features
from src.pipeline import compute_sample_weights
from src.model import (
    train_model, get_feature_columns, temporal_train_test_split,
    get_feature_importance, prune_features,
)
from src.export import export_to_onnx, validate_onnx_export
from src.source_model import (
    compute_source_features, train_source_model,
    SOURCE_FEATURE_COLS,
)

In [2]:
requests = pd.read_csv("data/http_requests.csv", parse_dates=["timestamp"])
headers = pd.read_csv("data/request_headers.csv")
labels = pd.read_csv("data/incident_labels.csv", parse_dates=["active_from", "active_until", "labeled_at"])

df = join_labels(requests, labels)
df = compute_per_request_features(df, headers)
df = compute_session_features(df)
df["sample_weight"] = compute_sample_weights(df)

train_df, test_df = temporal_train_test_split(df, test_date="2025-01-10")
feature_cols = get_feature_columns(df)

X_train = train_df[feature_cols].astype(float)
y_train = train_df["is_malicious"].astype(int)
w_train = train_df["sample_weight"]
X_test = test_df[feature_cols].astype(float)

# --- Tier 1: Request-level model ---
lgbm_params = {
    "n_estimators": 589, "num_leaves": 66, "max_depth": 8,
    "learning_rate": 0.0380, "subsample": 0.6867,
    "colsample_bytree": 0.6411, "min_child_samples": 44,
    "scale_pos_weight": 138.27, "reg_alpha": 0.0356,
    "reg_lambda": 0.00124, "verbose": -1, "metric": "average_precision",
}
model_full = train_model("lgbm", lgbm_params, X_train, y_train, w_train)
print(f"Tier 1 (request-level): {len(feature_cols)} features, {lgbm_params['n_estimators']} trees, max_depth={lgbm_params['max_depth']}")

# --- Tier 2: Source-level model ---
cutoff = pd.Timestamp("2025-01-10").date()
agg = compute_source_features(df, min_requests=2)
train_agg = agg[agg["day"] < cutoff]
test_agg = agg[agg["day"] >= cutoff].copy()

X_train_s = train_agg[SOURCE_FEATURE_COLS].astype(float)
y_train_s = train_agg["is_malicious"].astype(int)
X_test_s = test_agg[SOURCE_FEATURE_COLS].astype(float)

source_model = train_source_model(X_train_s, y_train_s)
print(f"Tier 2 (source-level):  {len(SOURCE_FEATURE_COLS)} features, 300 trees, max_depth=6")

Tier 1 (request-level): 36 features, 589 trees, max_depth=8
Tier 2 (source-level):  13 features, 300 trees, max_depth=6


## 1. ONNX Export & Validation

Export both tier models to ONNX format and validate predictions match the originals.

In [3]:
os.makedirs("edge-inference/model", exist_ok=True)

# Tier 1: Request-level ONNX
onnx_full_path = "edge-inference/model/model_full.onnx"
export_to_onnx(model_full, feature_cols, onnx_full_path)
val_full = validate_onnx_export(model_full, onnx_full_path, X_test)

# Tier 2: Source-level ONNX
onnx_source_path = "edge-inference/model/source_model.onnx"
export_to_onnx(source_model, SOURCE_FEATURE_COLS, onnx_source_path)
val_source = validate_onnx_export(source_model, onnx_source_path, X_test_s)

print("ONNX Validation:")
print(f"  Tier 1 (request) — max abs error: {val_full['max_abs_error']:.2e}, match: {val_full['match']}")
print(f"  Tier 2 (source)  — max abs error: {val_source['max_abs_error']:.2e}, match: {val_source['match']}")
print(f"\nFile size:")
print(f"  Tier 1 ONNX: {os.path.getsize(onnx_full_path) / 1024:.1f} KB")
print(f"  Tier 2 ONNX: {os.path.getsize(onnx_source_path) / 1024:.1f} KB")

ONNX Validation:
  Tier 1 (request) — max abs error: 4.98e-07, match: True
  Tier 2 (source)  — max abs error: 1.48e-07, match: True

File size:
  Tier 1 ONNX: 766.7 KB
  Tier 2 ONNX: 384.3 KB


2026-07-18 03:18:38.585 Python[4860:1386561] 2026-07-18 03:18:38.580161 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {21143} for output label
2026-07-18 03:18:38.753 Python[4860:1386561] 2026-07-18 03:18:38.753183 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {2452} for output label


In [4]:
# Tier 1: Request-level sample input
sample_row = X_test[feature_cols].iloc[0].astype(float).to_dict()
sample_input = {
    "feature_names": feature_cols,
    "values": [float(sample_row[f]) for f in feature_cols],
}
with open("edge-inference/model/sample_input.json", "w") as f:
    json.dump(sample_input, f, indent=2)
print(f"Tier 1 sample input saved: {len(feature_cols)} features")

# Tier 2: Source-level sample input
source_row = X_test_s.iloc[0].astype(float).to_dict()
source_sample = {
    "feature_names": list(SOURCE_FEATURE_COLS),
    "values": [float(source_row[f]) for f in SOURCE_FEATURE_COLS],
}
with open("edge-inference/model/source_sample_input.json", "w") as f:
    json.dump(source_sample, f, indent=2)
print(f"Tier 2 sample input saved: {len(SOURCE_FEATURE_COLS)} features")

Tier 1 sample input saved: 36 features
Tier 2 sample input saved: 13 features


## 2. Inference Latency Benchmarks

Measure per-request latency for single and batch inference across both tiers:
- LightGBM native Python
- ONNX Runtime Python

Each benchmark runs 1000 iterations for single-request, reporting p50/p95/p99.

In [5]:
def benchmark_latency(predict_fn, X_input, n_iterations=1000):
    """Run predict_fn n_iterations times, return latency stats in ms."""
    # Warmup
    for _ in range(10):
        predict_fn(X_input)

    latencies = []
    for _ in range(n_iterations):
        start = time.perf_counter()
        predict_fn(X_input)
        elapsed = (time.perf_counter() - start) * 1000  # ms
        latencies.append(elapsed)

    latencies = np.array(latencies)
    return {
        "mean_ms": float(np.mean(latencies)),
        "p50_ms": float(np.percentile(latencies, 50)),
        "p95_ms": float(np.percentile(latencies, 95)),
        "p99_ms": float(np.percentile(latencies, 99)),
        "min_ms": float(np.min(latencies)),
        "max_ms": float(np.max(latencies)),
    }

In [6]:
# ONNX sessions
sess_full = ort.InferenceSession(onnx_full_path)
input_name_full = sess_full.get_inputs()[0].name
sess_source = ort.InferenceSession(onnx_source_path)
input_name_source = sess_source.get_inputs()[0].name

results = {}
batch_sizes = [1, 10, 100, 1000]

# --- Tier 1: Request-level benchmarks ---
for batch_size in batch_sizes:
    batch_full = X_test[feature_cols].iloc[:batch_size].astype(float)
    n_iter = 1000 if batch_size <= 100 else 100

    def lgbm_full_fn(x, _m=model_full): return _m.predict_proba(x)
    def onnx_full_fn(x, _s=sess_full, _n=input_name_full):
        return _s.run(None, {_n: x.values.astype(np.float32)})

    results[f"lgbm_req_batch{batch_size}"] = benchmark_latency(lgbm_full_fn, batch_full, n_iter)
    results[f"onnx_req_batch{batch_size}"] = benchmark_latency(onnx_full_fn, batch_full, n_iter)

print("Tier 1 (request-level) benchmarks done.")

# --- Tier 2: Source-level benchmarks ---
for batch_size in [1, 10, 100]:
    n_src = min(batch_size, len(X_test_s))
    batch_source = X_test_s.iloc[:n_src].astype(float)
    n_iter = 1000 if n_src <= 100 else 100

    def lgbm_src_fn(x, _m=source_model): return _m.predict_proba(x)
    def onnx_src_fn(x, _s=sess_source, _n=input_name_source):
        return _s.run(None, {_n: x.values.astype(np.float32)})

    results[f"lgbm_src_batch{batch_size}"] = benchmark_latency(lgbm_src_fn, batch_source, n_iter)
    results[f"onnx_src_batch{batch_size}"] = benchmark_latency(onnx_src_fn, batch_source, n_iter)

print("Tier 2 (source-level) benchmarks done.")

2026-07-18 03:18:39.921 Python[4860:1386561] 2026-07-18 03:18:39.921585 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {10} for output label
2026-07-18 03:18:39.922 Python[4860:1386561] 2026-07-18 03:18:39.921956 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {10} for output label
2026-07-18 03:18:39.922 Python[4860:1386561] 2026-07-18 03:18:39.922287 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {10} for output label
2026-07-18 03:18:39.922 Python[4860:1386561] 2026-07-18 03:18:39.922427 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {10} for output label
2026-07-18 03:18:39.922 Python[4860:1386561] 2026-07-18 03:18:39.922551 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSizes] E

Tier 1 (request-level) benchmarks done.


2026-07-18 03:18:43.141 Python[4860:1386561] 2026-07-18 03:18:43.141413 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {10} for output label
2026-07-18 03:18:43.141 Python[4860:1386561] 2026-07-18 03:18:43.141688 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {10} for output label
2026-07-18 03:18:43.141 Python[4860:1386561] 2026-07-18 03:18:43.141977 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {10} for output label
2026-07-18 03:18:43.142 Python[4860:1386561] 2026-07-18 03:18:43.142120 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {10} for output label
2026-07-18 03:18:43.142 Python[4860:1386561] 2026-07-18 03:18:43.142225 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSizes] E

Tier 2 (source-level) benchmarks done.


2026-07-18 03:18:43.907 Python[4860:1386561] 2026-07-18 03:18:43.907497 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {100} for output label
2026-07-18 03:18:43.907 Python[4860:1386561] 2026-07-18 03:18:43.907752 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {100} for output label
2026-07-18 03:18:43.908 Python[4860:1386561] 2026-07-18 03:18:43.908161 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {100} for output label
2026-07-18 03:18:43.908 Python[4860:1386561] 2026-07-18 03:18:43.908473 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {100} for output label
2026-07-18 03:18:43.908 Python[4860:1386561] 2026-07-18 03:18:43.908678 [W:onnxruntime:, execution_frame.cc:913 VerifyOutputSize

In [7]:
n_feat_req = len(feature_cols)
n_feat_src = len(SOURCE_FEATURE_COLS)
rows = []

# Tier 1 results
for batch_size in batch_sizes:
    for runtime in ["lgbm", "onnx"]:
        key = f"{runtime}_req_batch{batch_size}"
        r = results[key]
        rows.append({
            "Model": f"Tier1 {runtime.upper()} ({n_feat_req}f)",
            "Batch": batch_size,
            "p50 (ms)": f"{r['p50_ms']:.3f}",
            "p95 (ms)": f"{r['p95_ms']:.3f}",
            "p99 (ms)": f"{r['p99_ms']:.3f}",
            "mean (ms)": f"{r['mean_ms']:.3f}",
        })

# Tier 2 results
for batch_size in [1, 10, 100]:
    for runtime in ["lgbm", "onnx"]:
        key = f"{runtime}_src_batch{batch_size}"
        r = results[key]
        rows.append({
            "Model": f"Tier2 {runtime.upper()} ({n_feat_src}f)",
            "Batch": batch_size,
            "p50 (ms)": f"{r['p50_ms']:.3f}",
            "p95 (ms)": f"{r['p95_ms']:.3f}",
            "p99 (ms)": f"{r['p99_ms']:.3f}",
            "mean (ms)": f"{r['mean_ms']:.3f}",
        })

latency_df = pd.DataFrame(rows)
print("Inference Latency — Two-Tier System (lower is better):")
print(latency_df.to_string(index=False))

Inference Latency — Two-Tier System (lower is better):
           Model  Batch p50 (ms) p95 (ms) p99 (ms) mean (ms)
Tier1 LGBM (36f)      1    0.515    0.552    0.621     0.519
Tier1 ONNX (36f)      1    0.027    0.031    0.051     0.026
Tier1 LGBM (36f)     10    0.564    0.597    0.624     0.567
Tier1 ONNX (36f)     10    0.098    0.116    0.132     0.101
Tier1 LGBM (36f)    100    0.788    0.891    0.954     0.804
Tier1 ONNX (36f)    100    0.375    0.487    0.552     0.375
Tier1 LGBM (36f)   1000    4.248    4.947    5.528     4.322
Tier1 ONNX (36f)   1000    3.802    4.136    4.308     3.827
Tier2 LGBM (13f)      1    0.461    0.596    0.694     0.476
Tier2 ONNX (13f)      1    0.018    0.020    0.036     0.019
Tier2 LGBM (13f)     10    0.492    0.520    0.566     0.495
Tier2 ONNX (13f)     10    0.065    0.069    0.077     0.066
Tier2 LGBM (13f)    100    0.647    0.759    1.925     0.691
Tier2 ONNX (13f)    100    0.179    0.234    0.265     0.186


## 3. Memory Footprint

Compare model sizes on disk (serialized) and peak runtime memory during inference.

In [8]:
import tempfile

# Disk size — joblib pkl
with tempfile.NamedTemporaryFile(suffix=".pkl") as f:
    joblib.dump(model_full, f.name)
    pkl_full_size = os.path.getsize(f.name)
with tempfile.NamedTemporaryFile(suffix=".pkl") as f:
    joblib.dump(source_model, f.name)
    pkl_source_size = os.path.getsize(f.name)

onnx_full_size = os.path.getsize(onnx_full_path)
onnx_source_size = os.path.getsize(onnx_source_path)

# Runtime memory — tracemalloc during inference
def measure_runtime_memory(predict_fn, X_input, n_runs=100):
    tracemalloc.start()
    for _ in range(n_runs):
        predict_fn(X_input)
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak

single_full = X_test[feature_cols].iloc[:1].astype(float)
single_source = X_test_s.iloc[:1].astype(float)

mem_lgbm_full = measure_runtime_memory(
    lambda x: model_full.predict_proba(x), single_full,
)
mem_onnx_full = measure_runtime_memory(
    lambda x: sess_full.run(None, {input_name_full: x.values.astype(np.float32)}),
    single_full,
)
mem_lgbm_source = measure_runtime_memory(
    lambda x: source_model.predict_proba(x), single_source,
)
mem_onnx_source = measure_runtime_memory(
    lambda x: sess_source.run(None, {input_name_source: x.values.astype(np.float32)}),
    single_source,
)

print("Memory Footprint — Two-Tier System:")
print(f"\n  Disk (serialized):")
print(f"    Tier 1 (request) — PKL: {pkl_full_size/1024:.1f} KB, ONNX: {onnx_full_size/1024:.1f} KB")
print(f"    Tier 2 (source)  — PKL: {pkl_source_size/1024:.1f} KB, ONNX: {onnx_source_size/1024:.1f} KB")
print(f"    Combined ONNX:     {(onnx_full_size + onnx_source_size)/1024:.1f} KB")
print(f"\n  Runtime peak memory (100 inferences):")
print(f"    Tier 1 LGBM:  {mem_lgbm_full/1024:.1f} KB")
print(f"    Tier 1 ONNX:  {mem_onnx_full/1024:.1f} KB")
print(f"    Tier 2 LGBM:  {mem_lgbm_source/1024:.1f} KB")
print(f"    Tier 2 ONNX:  {mem_onnx_source/1024:.1f} KB")

Memory Footprint — Two-Tier System:

  Disk (serialized):
    Tier 1 (request) — PKL: 1183.6 KB, ONNX: 766.7 KB
    Tier 2 (source)  — PKL: 627.4 KB, ONNX: 384.3 KB
    Combined ONNX:     1151.1 KB

  Runtime peak memory (100 inferences):
    Tier 1 LGBM:  136.9 KB
    Tier 1 ONNX:  11.3 KB
    Tier 2 LGBM:  139.8 KB
    Tier 2 ONNX:  1.8 KB


## 4. Throughput Feasibility

Calculate how many CPU cores are needed to sustain 50k req/s based on measured single-request p99 latency.

Formula: `threads_needed = target_rps × p99_latency_seconds`

In [9]:
target_rps = 50_000

print("Throughput Feasibility — Two-Tier System (target: 50,000 req/s):")
print(f"\n{'Model':<30} {'p99 (ms)':>10} {'Throughput/thread':>20} {'Threads for 50k':>18}")
print("-" * 80)

for label, key in [
    (f"Tier1 LGBM ({len(feature_cols)}f)", "lgbm_req_batch1"),
    (f"Tier1 ONNX ({len(feature_cols)}f)", "onnx_req_batch1"),
    (f"Tier2 LGBM ({len(SOURCE_FEATURE_COLS)}f)", "lgbm_src_batch1"),
    (f"Tier2 ONNX ({len(SOURCE_FEATURE_COLS)}f)", "onnx_src_batch1"),
]:
    p99 = results[key]["p99_ms"]
    throughput_per_thread = 1000.0 / p99
    threads_needed = target_rps * (p99 / 1000.0)
    print(f"{label:<30} {p99:>10.3f} {throughput_per_thread:>17.0f}/s {threads_needed:>17.1f}")

# Combined worst case: Tier 1 runs on every request, Tier 2 runs after ≥2 requests from same IP
t1_p99 = results["onnx_req_batch1"]["p99_ms"]
t2_p99 = results["onnx_src_batch1"]["p99_ms"]
print(f"\nCombined worst-case (both tiers sequential): {t1_p99 + t2_p99:.3f} ms")
print(f"  Tier 1 alone handles the critical path (every request)")
print(f"  Tier 2 runs asynchronously after IP accumulates ≥2 requests")
print(f"\nA typical edge node has 4-8 CPU cores.")
print(f"If threads_needed <= 8, the model fits within a single edge node's capacity.")

Throughput Feasibility — Two-Tier System (target: 50,000 req/s):

Model                            p99 (ms)    Throughput/thread    Threads for 50k
--------------------------------------------------------------------------------
Tier1 LGBM (36f)                    0.621              1610/s              31.0
Tier1 ONNX (36f)                    0.051             19535/s               2.6
Tier2 LGBM (13f)                    0.694              1441/s              34.7
Tier2 ONNX (13f)                    0.036             27998/s               1.8

Combined worst-case (both tiers sequential): 0.087 ms
  Tier 1 alone handles the critical path (every request)
  Tier 2 runs asynchronously after IP accumulates ≥2 requests

A typical edge node has 4-8 CPU cores.
If threads_needed <= 8, the model fits within a single edge node's capacity.


## 5. Summary

Key findings from Python benchmarks for the two-tier system. The Rust/WASM benchmarks in `edge-inference/` provide the production-representative numbers.

In [10]:
onnx_req_single = results["onnx_req_batch1"]
onnx_src_single = results["onnx_src_batch1"]

print("=" * 65)
print("EDGE DEPLOYMENT FEASIBILITY — TWO-TIER SYSTEM")
print("=" * 65)

print(f"\nTier 1 — Request-Level ONNX ({len(feature_cols)} features)")
print(f"  Single-request p99: {onnx_req_single['p99_ms']:.3f} ms")
print(f"  {'✓ PASS' if onnx_req_single['p99_ms'] < 5.0 else '✗ FAIL'}: under 5ms latency budget")
t1_threads = target_rps * (onnx_req_single["p99_ms"] / 1000)
print(f"  Threads for 50k req/s: {t1_threads:.1f}")
print(f"  {'✓ PASS' if t1_threads <= 8 else '✗ FAIL'}: fits in ≤8 cores")
print(f"  ONNX file: {onnx_full_size/1024:.1f} KB")
print(f"  Validation: max_abs_error = {val_full['max_abs_error']:.2e}")

print(f"\nTier 2 — Source-Level ONNX ({len(SOURCE_FEATURE_COLS)} features)")
print(f"  Single-request p99: {onnx_src_single['p99_ms']:.3f} ms")
print(f"  {'✓ PASS' if onnx_src_single['p99_ms'] < 5.0 else '✗ FAIL'}: under 5ms latency budget")
t2_threads = target_rps * (onnx_src_single["p99_ms"] / 1000)
print(f"  Threads for 50k req/s: {t2_threads:.1f}")
print(f"  {'✓ PASS' if t2_threads <= 8 else '✗ FAIL'}: fits in ≤8 cores")
print(f"  ONNX file: {onnx_source_size/1024:.1f} KB")
print(f"  Validation: max_abs_error = {val_source['max_abs_error']:.2e}")

print(f"\nCombined:")
print(f"  Total ONNX size: {(onnx_full_size + onnx_source_size)/1024:.1f} KB")
print(f"  Worst-case sequential: {onnx_req_single['p99_ms'] + onnx_src_single['p99_ms']:.3f} ms (still under 5ms)")
print(f"\nNext: run Rust/WASM benchmarks in edge-inference/")
print(f"  cargo run --release  (native)")
print(f"  cargo run --release -- model/source_model.onnx model/source_sample_input.json  (source)")

EDGE DEPLOYMENT FEASIBILITY — TWO-TIER SYSTEM

Tier 1 — Request-Level ONNX (36 features)
  Single-request p99: 0.051 ms
  ✓ PASS: under 5ms latency budget
  Threads for 50k req/s: 2.6
  ✓ PASS: fits in ≤8 cores
  ONNX file: 766.7 KB
  Validation: max_abs_error = 4.98e-07

Tier 2 — Source-Level ONNX (13 features)
  Single-request p99: 0.036 ms
  ✓ PASS: under 5ms latency budget
  Threads for 50k req/s: 1.8
  ✓ PASS: fits in ≤8 cores
  ONNX file: 384.3 KB
  Validation: max_abs_error = 1.48e-07

Combined:
  Total ONNX size: 1151.1 KB
  Worst-case sequential: 0.087 ms (still under 5ms)

Next: run Rust/WASM benchmarks in edge-inference/
  cargo run --release  (native)
  cargo run --release -- model/source_model.onnx model/source_sample_input.json  (source)
